In [ ]:
# Clone - Extract - Delete 
# GitHub metadata is optional - Commented here
import pandas as pd
import os
import subprocess
import shutil
import glob
from pathlib import Path
from dotenv import load_dotenv
import requests

# === LOAD GITHUB TOKEN ===
load_dotenv('All_tokens.env')
GITHUB_TOKEN = os.getenv('GITHUB_TOKEN')
if not GITHUB_TOKEN:
    raise ValueError("❌ GitHub token not found in All_tokens.env")

headers = {'Authorization': f'token {GITHUB_TOKEN}'}

# === CONFIGURATION ===
csv_path = r"C:\GitHub\Android-Mobile-Apps\8.1-Project_GitHub_URLs.csv"
clone_dir = Path(r"C:\GitHub\AndroidProjects\Cloned repos")
yml_output_dir = Path(r"C:\GitHub\AndroidProjects\Config Files")
commits_dir = Path(r"C:\GitHub\AndroidProjects\Commits")
build_info_dir = Path(r"C:\GitHub\AndroidProjects\BuildInfo")
# metadata_csv = Path(r"C:\GitHub\AndroidProjects\8.2-Project_Metadata.csv")  # Not used for now

# === CLEAN OLD DATA ===
for path in [clone_dir, yml_output_dir, commits_dir, build_info_dir]:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

# === LOAD AND CLEAN CSV ===
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip().str.lower()
df = df[df['github_url'].notna()]
df['github_url'] = df['github_url'].astype(str).str.strip()
df = df[df['github_url'].str.startswith("https://")]

# === PROCESS EACH REPO ===
for i, url in enumerate(df['github_url'], 1):
    parts = url.split('/')
    if len(parts) < 5:
        continue
    username, project = parts[-2], parts[-1].replace('.git', '')
    repo_name = f"{username}.{project}"
    repo_path = clone_dir / repo_name

    print(f"\n🔍 [{i}/{len(df)}] Processing {repo_name}...")

    # --- Shallow Clone ---
    try:
        subprocess.run(
            ['git', 'clone', '--depth', '1', url, str(repo_path)],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            timeout=300
        )
    except subprocess.TimeoutExpired:
        print(f"⏱️ Timeout while cloning {repo_name}, skipping...")
        continue

    # --- Extract .yml/.yaml Files ---
    for root, _, files in os.walk(repo_path):
        for file in files:
            if file.endswith(('.yml', '.yaml')):
                full_path = Path(root) / file
                rel_path = full_path.relative_to(repo_path)
                safe_name = f"{repo_name}.{str(rel_path).replace(os.sep, '_')}"
                shutil.copy2(full_path, yml_output_dir / safe_name)

    # --- Extract build.gradle & test lines ---
    info_path = build_info_dir / f"{repo_name}_build_info.txt"
    with open(info_path, 'w', encoding='utf-8') as out_file:
        for gradle_file in glob.glob(str(repo_path / '**/*.gradle*'), recursive=True):
            try:
                with open(gradle_file, 'r', encoding='utf-8', errors='ignore') as f:
                    lines = f.readlines()
                    test_lines = [line for line in lines if 'test' in line.lower()]
                    if test_lines:
                        out_file.write(f"\n--- {gradle_file} ---\n")
                        out_file.writelines(test_lines)
            except Exception:
                continue

    # --- Save Commit Log ---
    with open(commits_dir / f"{repo_name}_commits.txt", 'w', encoding='utf-8') as f:
        subprocess.run(['git', 'log', '--pretty=format:%h | %an | %ad | %s'],
                       cwd=repo_path, stdout=f, stderr=subprocess.DEVNULL)

    # --- Save Contributor List ---
    with open(commits_dir / f"{repo_name}_contributors.txt", 'w', encoding='utf-8') as f:
        subprocess.run(['git', 'shortlog', '-sn'],
                       cwd=repo_path, stdout=f, stderr=subprocess.DEVNULL)

    # === (OPTIONAL) GitHub API Metadata ===
    # Skipped to improve speed
    """
    api_url = f"https://api.github.com/repos/{username}/{project}"
    try:
        r = requests.get(api_url, headers=headers)
        if r.ok:
            data = r.json()
            # metadata extraction...
    except Exception as e:
        print(f"⚠️ Failed metadata for {repo_name}: {e}")
    """

    # --- Cleanup cloned repo ---
    shutil.rmtree(repo_path, ignore_errors=True)

print("\n✅ Finished processing all projects (without metadata).")



🔍 [1/1796] Processing AChep.15puzzle...
